# Member 1 - Logistic Regression Model

## Objective
The goal of this notebook is to train and evaluate a Logistic Regression model for the Obesity dataset.


## Step 1: Import libraries and configure the notebook environment

This cell prepares all dependencies used in the notebook.

- `os` and `sys` help the notebook find the project root and import modules from the shared folder.
- `time` and `tracemalloc` are used to measure runtime and memory usage so Member 1 can be compared with Member 2.
- Scikit-learn modules are imported for preprocessing, model building, training, and hyperparameter tuning.
- Shared helper functions are imported so preprocessing and evaluation stay consistent across the whole team.

Without this setup cell, the later cells would not be able to load data, build the model pipeline, or report results.

In [1]:
# Import modules for path handling, timing, and memory measurement.
import os
import sys
import time
import tracemalloc

# Build the absolute path to the project root from the current notebook location.
project_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
# Add the project root to Python's import path if it is not already available.
if project_root not in sys.path:
    sys.path.append(project_root)

# Import scikit-learn tools used for splitting data, preprocessing, modeling, and tuning.
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression

# Import shared project helpers for loading data and reporting evaluation metrics.
from shared.preprocessing.preprocess import load_data, split_features_target, get_feature_types
from shared.evaluation.metrics import evaluate_model, print_evaluation_results

## Step 2: Load the raw dataset

This cell loads the obesity dataset from the `data/raw` folder and performs the first split between features and target.

- `file_path` uses `project_root` so the notebook can locate the dataset reliably.
- `load_data(file_path)` reads the CSV file into a pandas DataFrame.
- `split_features_target(...)` separates the predictors into `X` and the obesity class into `y`.
- `df.head()` displays the first rows so we can visually confirm the dataset loaded correctly.

This step is the main data-loading checkpoint for the notebook.

In [2]:
# Build the full dataset path from the project root.
file_path = os.path.join(project_root, "data", "raw", "ObesityDataSet_raw_and_data_sinthetic.csv")
# Load the raw CSV file into a DataFrame.
df = load_data(file_path)
# Split the DataFrame into input features `X` and target labels `y`.
X, y = split_features_target(df, target_column="NObeyesdad")
# Show the first rows so we can inspect whether the data loaded correctly.
df.head()

,Gender,Age,Height,Weight,family_history_with_overweight,FAVC,FCVC,NCP,CAEC,SMOKE,CH2O,SCC,FAF,TUE,CALC,MTRANS,NObeyesdad
0,Female,21.0,1.62,64.0,yes,no,2.0,3.0,Sometimes,no,2.0,no,0.0,1.0,no,Public_Transportation,Normal_Weight
1,Female,21.0,1.52,56.0,yes,no,3.0,3.0,Sometimes,yes,3.0,yes,3.0,0.0,Sometimes,Public_Transportation,Normal_Weight
2,Male,23.0,1.80,77.0,yes,no,2.0,3.0,Sometimes,no,2.0,no,2.0,1.0,Frequently,Public_Transportation,Normal_Weight
3,Male,27.0,1.80,87.0,no,no,3.0,3.0,Sometimes,no,2.0,no,2.0,0.0,Frequently,Walking,Overweight_Level_I
4,Male,22.0,1.78,89.8,no,no,2.0,1.0,Sometimes,no,2.0,no,0.0,0.0,Sometimes,Public_Transportation,Overweight_Level_II


## Step 3: Encode the target and inspect feature types

This cell prepares the target variable and checks which feature columns are numeric or categorical.

- `LabelEncoder()` converts the target labels into numeric values because Logistic Regression needs numeric class labels.
- `get_feature_types(X)` scans the dataset and groups columns into numeric and categorical lists.
- Printing both lists helps verify that the next preprocessing step will apply the right transformation to each column.

This is an important preparation step before building the machine learning pipeline.

In [3]:
# Encode the target labels into numeric form so the classifier can train on them.
target_encoder = LabelEncoder()
y_encoded = target_encoder.fit_transform(y)
# Detect which feature columns are numeric and which are categorical.
numeric_features, categorical_features = get_feature_types(X)
# Print both groups to verify the preprocessing plan.
print('Numeric features:', numeric_features)
print('Categorical features:', categorical_features)

Numeric features: ['Age', 'Height', 'Weight', 'FCVC', 'NCP', 'CH2O', 'FAF', 'TUE']
Categorical features: ['Gender', 'family_history_with_overweight', 'FAVC', 'CAEC', 'SMOKE', 'SCC', 'CALC', 'MTRANS']


## Step 4: Build the preprocessing pipeline and tuning workflow

This cell creates the full training workflow for Member 1's Logistic Regression model.

- `train_test_split(...)` creates training and testing sets while preserving class balance.
- `ColumnTransformer(...)` lets us preprocess numeric and categorical columns differently.
- `StandardScaler()` is applied to numeric features because Logistic Regression usually performs better when numeric inputs are scaled.
- `OneHotEncoder(handle_unknown='ignore')` converts text categories into numeric indicator columns.
- `Pipeline(...)` combines preprocessing and the classifier into one reusable workflow.
- `param_grid` defines the hyperparameters we want to compare.
- `GridSearchCV(...)` trains multiple versions of the model and selects the best one using cross-validation.

This is the core machine learning construction step in the notebook.

In [4]:
# Split the full dataset into training and testing sets.
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

# Create a preprocessing object that scales numeric columns and one-hot encodes categorical ones.
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
    ]
)

# Build a pipeline so preprocessing and Logistic Regression are trained together.
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(max_iter=3000))
])

# Define the hyperparameter values to compare during grid search.
param_grid = {
    'model__C': [0.1, 1.0, 10.0],
    'model__solver': ['lbfgs']
}

# Create GridSearchCV to evaluate multiple model settings with cross-validation.
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

## Step 5: Train and tune the model

This cell fits the full pipeline on the training data and records how long training takes.

- `start_train` stores the timestamp before training begins.
- `grid_search.fit(X_train, y_train)` trains the preprocessing pipeline and Logistic Regression model for every parameter combination in the search grid.
- `train_time` calculates the total runtime of this training stage.
- `best_params_` shows which hyperparameters achieved the strongest validation result.
- `best_score_` reports the best cross-validation accuracy found by GridSearchCV.

This gives both model quality information and a runtime measurement for comparison.

In [5]:
# Record the time before model training starts.
start_train = time.time()
# Train and tune the pipeline on the training set.
grid_search.fit(X_train, y_train)
# Compute the total runtime of the training stage.
train_time = time.time() - start_train
# Display the best hyperparameters and validation score found by GridSearchCV.
print('Best Parameters:', grid_search.best_params_)
print('Best Cross-Validation Score:', round(grid_search.best_score_, 4))
# Print the measured training time.
print(f'Training time: {train_time:.4f} seconds')

Best Parameters: {'model__C': 10.0, 'model__solver': 'lbfgs'}
Best Cross-Validation Score: 0.936
Training time: 9.8313 seconds


## Step 6: Evaluate the best model and measure prediction cost

This cell retrieves the best tuned model, measures prediction runtime, estimates peak memory usage, and prints the final evaluation metrics.

- `best_model = grid_search.best_estimator_` selects the strongest trained pipeline.
- `time.time()` is used again to measure how long prediction on the test set takes.
- `tracemalloc` is used to estimate peak memory usage during a fresh training pass and during prediction.
- `evaluate_model(...)` computes the shared evaluation metrics, including accuracy, classification report, and confusion matrix.
- The printed outputs make it easier to compare Member 1's model with the workflows used by the other team members.

This is the final performance and evaluation checkpoint of the notebook.

In [6]:
# Retrieve the best trained pipeline from the completed grid search.
best_model = grid_search.best_estimator_
# Measure how long prediction on the test set takes.
start_pred = time.time()
y_pred = best_model.predict(X_test)
pred_time = time.time() - start_pred
print(f'Prediction time: {pred_time:.4f} seconds')

# Start memory tracking for a fresh training run.
tracemalloc.start()
# Recreate the grid search object so memory measurement reflects a clean fit.
mem_grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)
# Fit the fresh grid search object and capture the peak training memory.
mem_grid_search.fit(X_train, y_train)
_, peak_train_mem = tracemalloc.get_traced_memory()
tracemalloc.stop()

# Extract the best estimator from the memory-measurement run.
mem_best_model = mem_grid_search.best_estimator_
# Start memory tracking for the prediction stage.
tracemalloc.start()
# Run prediction to estimate the peak memory used during inference.
_ = mem_best_model.predict(X_test)
_, peak_pred_mem = tracemalloc.get_traced_memory()
tracemalloc.stop()

# Print peak memory usage for both training and prediction.
print(f'Peak memory usage during training: {peak_train_mem / 1024:.2f} KB')
print(f'Peak memory usage during prediction: {peak_pred_mem / 1024:.2f} KB')

# Evaluate the tuned model on the held-out test set and reuse the timed predictions.
results = evaluate_model(best_model, X_test, y_test)
results['y_pred'] = y_pred
# Print the full evaluation summary and final test accuracy.
print_evaluation_results(results)
print(f'\nTest Accuracy: {results["accuracy"]:.4f}')

Prediction time: 0.0130 seconds
Peak memory usage during training: 1052.74 KB
Peak memory usage during prediction: 261.29 KB
Accuracy: 0.9290780141843972

Classification Report:
              precision    recall  f1-score   support

           0       0.95      1.00      0.97        54
           1       0.86      0.83      0.84        58
           2       0.96      0.94      0.95        70
           3       0.95      0.98      0.97        60
           4       1.00      0.98      0.99        65
           5       0.84      0.84      0.84        58
           6       0.93      0.91      0.92        58

    accuracy                           0.93       423
   macro avg       0.93      0.93      0.93       423
weighted avg       0.93      0.93      0.93       423


Confusion Matrix:
[[54  0  0  0  0  0  0]
 [ 3 48  0  0  0  7  0]
 [ 0  0 66  1  0  0  3]
 [ 0  0  1 59  0  0  0]
 [ 0  0  0  1 64  0  0]
 [ 0  8  0  0  0 49  1]
 [ 0  0  2  1  0  2 53]]

Test Accuracy: 0.9291


## Result Interception
The Logistic Regression model is used as a baseline classification model.

### Strengths
- simple and fast
- easy to train
- easy to explain
- good as a baseline model for comparison

### Weaknesses
- may not capture complex nonlinear relationships
- may perform worse than ensemble models such as Random Forest